# 13 — Error Handling and Retries

Every tool accepts retry-related parameters (see [`docs/05_tools_reference.md`](../docs/05_tools_reference.md)): maximum retries, retry delay, retry delay maximum, retry mode (`fixed`/`exponential`), and a network timeout. This notebook shows how to apply the same ideas on the *client* side too.

In [ ]:
import asyncio
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
from src.mcp_client import connect

async def call_with_backoff(namespace, tool, args=None, max_retries=3, base_delay=1.0):
    last_exc = None
    for attempt in range(max_retries):
        try:
            async with connect(namespaces=[namespace], read_only=True) as client:
                return await client.call_tool(tool, args or {})
        except Exception as exc:
            last_exc = exc
            delay = base_delay * (2 ** attempt)
            print(f"Attempt {attempt + 1} failed ({exc}); retrying in {delay:.1f}s")
            await asyncio.sleep(delay)
    raise last_exc